In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("ModelComparison") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

df_final = spark.read.parquet('data/features/final_dataset')
df_final = df_final.filter(col('product_id').isNotNull())
df_final = df_final.fillna(-1, subset=['favorite_department', 'favorite_aisles'])
df_final = df_final.fillna(0)

# Balance
df_positive = df_final.filter(col('label') == 1)
df_negative = df_final.filter(col('label') == 0).sample(fraction=0.12, seed=42)
df_balanced = df_positive.union(df_negative)

# Convert to pandas
df_pd = df_balanced.toPandas()
print(df_pd.shape)

(548735, 20)


In [3]:
import subprocess
subprocess.run(['pip', 'install', 'xgboost', 'lightgbm'], capture_output=True)

CompletedProcess(args=['pip', 'install', 'xgboost', 'lightgbm'], returncode=0, stdout=b'Collecting xgboost\n  Downloading xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)\nCollecting lightgbm\n  Downloading lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl.metadata (17 kB)\nRequirement already satisfied: numpy in /opt/conda/lib/python3.11/site-packages (from xgboost) (1.24.4)\nCollecting nvidia-nccl-cu12 (from xgboost)\n  Downloading nvidia_nccl_cu12-2.30.4-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)\nRequirement already satisfied: scipy in /opt/conda/lib/python3.11/site-packages (from xgboost) (1.11.3)\nDownloading xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl (131.7 MB)\n\x1b[?25l   \x1b\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94\x81\xe2\x94

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
import xgboost as xgb
import lightgbm as lgb
import time

feature_cols = ['total_orders','avg_days_between_orders','avg_basket_size','overall_reorder_rate',
               'favorite_hour','favorite_day','favorite_department','favorite_aisles',
               'product_total_orders','product_reorder_rate','product_unique_users',
               'up_times_bought','up_reorder_rate','up_avg_cart_pos','up_last_order',
               'orders_since_last_buy','up_order_rate']

X = df_pd[feature_cols]
y = df_pd['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost
t0 = time.time()
xgb_model = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - t0
xgb_pred = xgb_model.predict(X_test)
print(f"XGBoost — Train: {xgb_time:.1f}s, F1: {f1_score(y_test, xgb_pred):.4f}, AUC: {roc_auc_score(y_test, xgb_pred):.4f}")

# LightGBM
t0 = time.time()
lgb_model = lgb.LGBMClassifier(n_estimators=100, random_state=42)
lgb_model.fit(X_train, y_train)
lgb_time = time.time() - t0
lgb_pred = lgb_model.predict(X_test)
print(f"LightGBM — Train: {lgb_time:.1f}s, F1: {f1_score(y_test, lgb_pred):.4f}, AUC: {roc_auc_score(y_test, lgb_pred):.4f}")

XGBoost — Train: 15.1s, F1: 0.6916, AUC: 0.6863
[LightGBM] [Info] Number of positive: 218189, number of negative: 220799
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024304 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2406
[LightGBM] [Info] Number of data points in the train set: 438988, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.497027 -> initscore=-0.011891
[LightGBM] [Info] Start training from score -0.011891
LightGBM — Train: 2.7s, F1: 0.6901, AUC: 0.6846


In [5]:
import numpy as np

# Single prediction speed
sample = X_test.iloc[:1]

t0 = time.time()
for _ in range(1000):
    xgb_model.predict_proba(sample)
xgb_inf = (time.time() - t0) / 1000
print(f"XGBoost single inference: {xgb_inf*1000:.2f}ms")

t0 = time.time()
for _ in range(1000):
    lgb_model.predict_proba(sample)
lgb_inf = (time.time() - t0) / 1000
print(f"LightGBM single inference: {lgb_inf*1000:.2f}ms")

XGBoost single inference: 9.92ms
LightGBM single inference: 1.46ms


In [11]:
import pickle
with open('/home/jovyan/instacart-reorder-prediction/models/lgb_model.pkl', 'wb') as f:
    pickle.dump(lgb_model, f)
print('LightGBM model saved!')

LightGBM model saved!


In [12]:
import os
os.path.exists('/home/jovyan/instacart-reorder-prediction/models/lgb_model.pkl')

True

In [13]:
import subprocess
result = subprocess.run(['find', '/home/jovyan', '-name', 'lgb_model.pkl'], capture_output=True, text=True)
print(result.stdout)

/home/jovyan/instacart-reorder-prediction/models/lgb_model.pkl

